# Sign Language Recognition
Run cells 1 to 6 in order. Allow camera access when prompted.

**Before starting:** Runtime > Change runtime type > T4 GPU

In [ ]:
# Cell 1 - Install dependencies
!pip install mediapipe tensorflow scikit-learn opencv-python-headless -q
print('Done')

In [ ]:
# Cell 2 - Clone repo
!git clone https://github.com/Gtblaster/final-project-of-6th-sem.git
%cd final-project-of-6th-sem
!mkdir -p data
print('Ready')

In [ ]:
# Cell 3 - Setup camera + MediaPipe
import os, csv, urllib.request, time
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

# Download hand landmarker model
MODEL_PATH = 'hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Downloading hand landmarker model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
        MODEL_PATH
    )
    print('Downloaded')

base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
hand_options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.70,
    min_hand_presence_confidence=0.70,
    min_tracking_confidence=0.70,
    running_mode=mp_vision.RunningMode.IMAGE
)
landmarker = mp_vision.HandLandmarker.create_from_options(hand_options)

# JavaScript to capture one frame from browser camera
JS_CODE = """
async function snap() {
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  const video  = document.createElement('video');
  video.srcObject = stream;
  await new Promise(r => { video.onloadedmetadata = r; });
  video.play();
  await new Promise(r => setTimeout(r, 500));
  const canvas = document.createElement('canvas');
  canvas.width  = video.videoWidth;
  canvas.height = video.videoHeight;
  canvas.getContext('2d').drawImage(video, 0, 0);
  stream.getTracks().forEach(t => t.stop());
  return canvas.toDataURL('image/jpeg', 0.8);
}
snap();
"""

def capture_frame():
    data_url  = eval_js(JS_CODE)
    img_bytes = b64decode(data_url.split(',')[1])
    img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
    return cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

def normalize_landmarks(lms, handedness):
    wrist  = np.array([lms[0].x, lms[0].y, lms[0].z], dtype=np.float32)
    coords = np.array([[lm.x, lm.y, lm.z] for lm in lms], dtype=np.float32)
    coords -= wrist
    if handedness.lower() == 'left':
        coords[:, 0] *= -1.0
    flat  = coords.flatten()
    scale = np.max(np.abs(flat)) + 1e-6
    return flat / scale

def get_landmarks(frame):
    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_img)
    if not result.hand_landmarks:
        return None, None
    lms  = result.hand_landmarks[0]
    side = result.handedness[0][0].category_name
    xs   = [lm.x for lm in lms]
    ys   = [lm.y for lm in lms]
    area = (max(xs) - min(xs)) * (max(ys) - min(ys))
    if area < 0.015:
        return None, None
    return normalize_landmarks(lms, side), side

print('Setup complete')

In [ ]:
# Cell 4 - Collect landmark data
from IPython.display import clear_output

CLASSES     = ['Hello', 'ILoveYou', 'No', 'Please', 'Thanks', 'Yes']
SAMPLES_PER = 150
OUTPUT_CSV  = 'data/landmarks.csv'

with open(OUTPUT_CSV, 'w', newline='') as f:
    header = [f'{ax}{i}' for i in range(21) for ax in ['x','y','z']] + ['label']
    csv.writer(f).writerow(header)

for cls in CLASSES:
    print(f'Next sign: {cls}')
    print('Hold the sign in front of camera. Starting in 3 seconds...')
    time.sleep(3)
    collected = 0
    missed    = 0
    while collected < SAMPLES_PER:
        frame = capture_frame()
        feat, side = get_landmarks(frame)
        if feat is not None:
            with open(OUTPUT_CSV, 'a', newline='') as f:
                csv.writer(f).writerow(feat.tolist() + [cls])
            collected += 1
            missed = 0
        else:
            missed += 1
        clear_output(wait=True)
        print(f'Class: {cls}  |  {collected}/{SAMPLES_PER}')
        if missed > 5:
            print('Hand not detected - move closer or improve lighting')
    print(f'{cls} done')

print('All data collected. Run Cell 5 to train.')

In [ ]:
# Cell 5 - Train model
!python train_model.py

In [ ]:
# Cell 6 - Live inference
import tensorflow as tf
import matplotlib.pyplot as plt
from collections import deque
from IPython.display import clear_output

WINDOW_SIZE = 15
MIN_CONF    = 0.70
HOLD_FRAMES = 8
RUN_FRAMES  = 300

sign_model = tf.keras.models.load_model('data/sign_model.keras')
classes    = np.load('data/label_classes.npy', allow_pickle=True)
print(f'Model loaded. Classes: {classes}')

COLORS = {
    'Hello': '#00C832', 'ILoveYou': '#C800C8',
    'No': '#DC0032',    'Please': '#C88C00',
    'Thanks': '#00B4B4','Yes': '#008CFF'
}

buf           = deque(maxlen=WINDOW_SIZE)
candidate     = None
candidate_cnt = 0
displayed     = None
disp_conf     = 0.0

for i in range(RUN_FRAMES):
    frame = capture_frame()
    feat, side = get_landmarks(frame)

    if feat is not None:
        buf.append(feat)
        if len(buf) == WINDOW_SIZE:
            seq   = np.array(buf, dtype=np.float32)[np.newaxis]
            probs = sign_model.predict(seq, verbose=0)[0]
            top_i = int(np.argmax(probs))
            top_p = float(probs[top_i])
            pred  = classes[top_i]
            if top_p >= MIN_CONF:
                if pred == candidate:
                    candidate_cnt += 1
                else:
                    candidate     = pred
                    candidate_cnt = 1
                if candidate_cnt >= HOLD_FRAMES:
                    displayed = candidate
                    disp_conf = top_p

            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            ax1.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax1.axis('off')
            title = f'{displayed}  {disp_conf:.0%}' if displayed else 'Detecting...'
            color = COLORS.get(displayed, 'white') if displayed else 'red'
            ax1.set_title(title, fontsize=22, fontweight='bold', color=color)
            bar_colors = [COLORS.get(c, '#888888') for c in classes]
            ax2.barh(classes, probs, color=bar_colors)
            ax2.set_xlim(0, 1)
            ax2.set_title('Confidence')
            for j, p in enumerate(probs):
                ax2.text(min(p + 0.02, 0.92), j, f'{p:.0%}', va='center')
            plt.tight_layout()
            plt.show()
    else:
        buf.clear()
        candidate = None
        candidate_cnt = 0
        displayed = None
        clear_output(wait=True)
        print(f'Frame {i}: No hand detected')

print('Done')